# ADP snapshots and movement

- **Question:** Given only market observations available at a stated timestamp, how has a player's average draft position (ADP) moved and what do transparent baselines predict at a future horizon?
- **Data:** Immutable FFC or manually imported ESPN ADP captures whose raw SHA-256 hashes match their manifests, normalized into the canonical DuckDB tables.
- **Unit of observation:** One source player row in one market snapshot, scoped by source, season, scoring format, team count, and position filter.
- **Target:** Future average pick at an explicit forecast timestamp. That target is not yet observable for the current one-snapshot archive.
- **Feature cutoff:** A feature or baseline at time `t` may use only snapshots with `captured_at <= t`; later captures are excluded.
- **Validation:** Expanding chronological evaluation after enough independent dates exist; never a random row split. The current one-snapshot archive supports persistence but cannot validate a trend.
- **Scope boundary:** ADP movement describes the market, not player quality. This notebook does not rank players, recommend a pick, or simulate a draft.

## Build the local Phase 5 publication first

Run these idempotent commands from the repository root when the canonical tables are not ready:

```powershell
fantasy-draft data load-adp
fantasy-draft models build-adp-baselines
fantasy-draft data audit
```

The loader verifies manifest hashes and never rewrites raw captures. The cells below use package APIs and read-only DuckDB connections; they print a prerequisite message instead of creating data.

In [ ]:
from __future__ import annotations

from datetime import UTC, datetime

import duckdb
import pandas as pd

from fantasy_draft_ai.config import find_project_root, load_config
from fantasy_draft_ai.services.adp_market import adp_market_status


def table_names(connection: duckdb.DuckDBPyConnection) -> set[str]:
    return {
        str(row[0])
        for row in connection.execute(
            "SELECT table_name FROM information_schema.tables WHERE table_schema = 'main'"
        ).fetchall()
    }


PROJECT_ROOT = find_project_root()
app_config = load_config()
WAREHOUSE_PATH = app_config.resolve(app_config.paths.warehouse)
market_status = adp_market_status(app_config)
print({
    "warehouse": str(WAREHOUSE_PATH),
    "phase5_available": market_status.available,
    "status": market_status.message,
})

## Verify snapshot identity and immutable lineage

A snapshot is an independent market observation only when its capture timestamp and content identify a distinct raw capture. Duplicate manifests for the same raw file must not inflate the history. Canonical player IDs may remain null; unresolved rows stay keyed by source ID rather than being joined by display name.

In [ ]:
snapshot_metadata = pd.DataFrame()
if not WAREHOUSE_PATH.is_file():
    print("Warehouse not found. Run the Phase 5 prerequisite commands above.")
else:
    with duckdb.connect(str(WAREHOUSE_PATH), read_only=True) as connection:
        available_tables = table_names(connection)
        if "adp_snapshot_metadata" not in available_tables:
            print("ADP snapshot tables are not initialized. Run: fantasy-draft data load-adp")
        else:
            snapshot_metadata = connection.execute(
                """
                SELECT snapshot_id, source, captured_at, season, scoring_format,
                       team_count, position_scope, raw_sha256, row_count
                FROM adp_snapshot_metadata
                ORDER BY captured_at, source, snapshot_id
                """
            ).df()
snapshot_metadata

## Prove the cutoff rule with the production API

This tiny, clearly synthetic series demonstrates the time contract. At the second timestamp, the third observation is invisible: persistence is available, while both trend methods remain unavailable. At the third timestamp, all three dated observations are eligible and the trend baselines can activate. The fixture teaches mechanics only and is never written to the warehouse.

In [ ]:
from fantasy_draft_ai.models.adp.movement import (
    AdpIdentity,
    AdpObservation,
    movement_baselines_as_of,
    movement_features_as_of,
)

identity = AdpIdentity(source="fixture", raw_source_row_id="WR_EXAMPLE")
fixture_observations = (
    AdpObservation(identity, datetime(2026, 7, 1, tzinfo=UTC), 50.0),
    AdpObservation(identity, datetime(2026, 7, 4, tzinfo=UTC), 47.0),
    AdpObservation(identity, datetime(2026, 7, 8, tzinfo=UTC), 43.0),
)

second_cutoff = fixture_observations[1].captured_at
third_cutoff = fixture_observations[2].captured_at
second_features = movement_features_as_of(
    fixture_observations, cutoff_at=second_cutoff
)[0]
second_forecasts = movement_baselines_as_of(
    fixture_observations, cutoff_at=second_cutoff, horizon_days=1
)
third_forecasts = movement_baselines_as_of(
    fixture_observations, cutoff_at=third_cutoff, horizon_days=1
)

assert second_features.observation_count == 2
assert second_features.current_adp == 47.0
assert [forecast.status for forecast in second_forecasts] == [
    "available", "unavailable", "unavailable"
]
assert all(forecast.status == "available" for forecast in third_forecasts)

pd.DataFrame(
    [
        {
            "cutoff": cutoff.date(),
            "method": forecast.method,
            "status": forecast.status,
            "history_count": forecast.training_observation_count,
            "predicted_adp": forecast.predicted_adp,
        }
        for cutoff, forecasts in (
            (second_cutoff, second_forecasts),
            (third_cutoff, third_forecasts),
        )
        for forecast in forecasts
    ]
)

## Inspect the persisted movement publication

The query keeps movement separate from player projections. Missing changes and null trend predictions mean insufficient history; they are not zeros. Positive change means a player moved later in drafts, while negative change means the market price became earlier.

In [ ]:
movement_preview = pd.DataFrame()
required_tables = {
    "adp_snapshots",
    "adp_movement_features",
    "adp_movement_forecasts",
}
if WAREHOUSE_PATH.is_file():
    with duckdb.connect(str(WAREHOUSE_PATH), read_only=True) as connection:
        if required_tables <= table_names(connection):
            movement_preview = connection.execute(
                """
                SELECT snapshot.player_name, snapshot.position, snapshot.source,
                       snapshot.captured_at, snapshot.average_pick,
                       movement.prior_average_pick, movement.change_7d,
                       movement.velocity_per_day, movement.observation_count,
                       persistence.predicted_average_pick AS persistence_adp,
                       linear.status AS linear_status,
                       linear.predicted_average_pick AS linear_adp,
                       weighted.status AS ew_status,
                       weighted.predicted_average_pick AS ew_adp,
                       snapshot.mapping_confidence
                FROM adp_snapshots AS snapshot
                JOIN adp_movement_features AS movement
                  USING (snapshot_id, raw_source_row_id)
                JOIN adp_movement_forecasts AS persistence
                  ON persistence.snapshot_id = snapshot.snapshot_id
                 AND persistence.raw_source_row_id = snapshot.raw_source_row_id
                 AND persistence.baseline_name = 'persistence'
                JOIN adp_movement_forecasts AS linear
                  ON linear.snapshot_id = snapshot.snapshot_id
                 AND linear.raw_source_row_id = snapshot.raw_source_row_id
                 AND linear.baseline_name = 'linear_trend'
                JOIN adp_movement_forecasts AS weighted
                  ON weighted.snapshot_id = snapshot.snapshot_id
                 AND weighted.raw_source_row_id = snapshot.raw_source_row_id
                 AND weighted.baseline_name = 'exponentially_weighted_trend'
                ORDER BY snapshot.average_pick, snapshot.source, snapshot.player_name
                LIMIT 15
                """
            ).df()
        else:
            print("Movement tables are not built. Run: fantasy-draft models build-adp-baselines")
movement_preview

In [ ]:
capability_summary = pd.DataFrame(
    [
        ("production snapshots", market_status.snapshot_count),
        ("ADP observations", market_status.observation_rows),
        ("persistence forecasts ready", market_status.persistence_ready_rows),
        ("linear forecasts ready", market_status.linear_ready_rows),
        ("EW forecasts ready", market_status.ew_ready_rows),
        ("calibration status", market_status.calibration_status),
        ("supervised status", market_status.supervised_status),
    ],
    columns=["capability", "value"],
)
capability_summary

## Interpretation boundary

With one independent production capture, every player has only one dated market point. Persistence can repeat that point, but a slope, acceleration, trend score, and chronological error estimate are not supported. Keep acquiring immutable snapshots; only later captures can evaluate forecasts made from earlier cutoffs. Nothing in this notebook is a signal to draft or avoid a player.